# Exercise 18 - Ollama, RAG and MCP Server

Estimated time: **45-50 minutes**

**This exercise is divided into 3 parts:**

Part A: Install and run Gemma3 model using Ollama

Part B: Implementation of a Text based RAG using Ollama, Langchain and Chroma

Part C: AI Code Intrepreter agent using Mistral model and APIs

- Recommended Hardware accelerator: **CPU or GPU**

We use code and APIs from the following sources:

1. Google AI for Developers guide, [Run Gemma with Ollama](https://ai.google.dev/gemma/docs/integrations/ollama)
2. [Retrieval-augmented generation](https://keras.io/keras_hub/guides/rag_pipeline_with_keras_hub/) (RAG) pipeline with KerasHub
3. AI Code Interpreter Agent from [Mistral](https://docs.mistral.ai/agents/connectors/code_interpreter/)

### **Part A: Run Gemma3 with Ollama**

### Get and install Ollama on Kaggle VM

In [ ]:
!sudo apt-get install -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

- Start Ollama Server in the background using Python *subprocess*.

- Keep the Terminal window open to observe how Ollama responds to commands using HTTP protocol

In [ ]:
import subprocess
import time

# Start the Ollama server in the background
process = subprocess.Popen("ollama serve", shell=True)

# Give it a few seconds to initialize
time.sleep(5)

- Obtain Ollama version

In [ ]:
!ollama --version

- Get a local copy of the open-weight Google model Gemma 3.  You can specify the Gemma3 model size by using the options, :270m, 1b, 4b, 12b, or 27 b.  The suffix **m** specifies million while **b** is for billion parameters.
- We pull the a moderately sized model which has 1 billion parameters
- Multiple models can be pulled based on the available memory
- You can view the entire [library](https://ollama.com/library) of open-weight models accessible through Ollama

In [ ]:
!ollama pull gemma3:1b

- Check locally available Ollama list of models.  The pulled Gemma 3 model should appear in the list

In [ ]:
!ollama list

Execute Gemma3 with prompt *"What makes the outdoors so beautiful*" as a CLI and as a REST API.
- Ollama uses port 11434 on localhost
- Output from the model is stored in Markdown file *outdoors.md*
- **The model takes some time to process the prompt**
- Once the processing is complete, we open the output file in the next cell

In [ ]:
!ollama run gemma3:1b "What makes the outdoors so beautiful?"  > outdoors.md

In [ ]:
from IPython.display import Markdown

# Read the content from the file
with open('outdoors.md', 'r') as f:
    outdoors_content = f.read()

Markdown(outdoors_content)

In [ ]:
import requests
from IPython.display import Markdown
import json

url = "http://localhost:11434/api/generate"
payload = {
    "model": "gemma3:1b",
    "prompt": "What makes the beaches so relaxing?"
}

response = requests.post(url, json=payload, stream=True)

beaches_content = ""
for chunk in response.iter_content(chunk_size=None):
    if chunk:
        try:
            json_chunk = chunk.decode('utf-8')
            data = json.loads(json_chunk)
            beaches_content += data.get("response", "")
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON chunk: {e}")
            print(f"Problematic chunk: {chunk}")
            beaches_content += chunk.decode('utf-8') # Append raw chunk if JSON decoding fails


# Display the accumulated content as Markdown
Markdown(beaches_content)

### **Part B. Text based RAG using Ollama, Langchain and ChromaDB**

Install the following Python Packages,
- Langchain and Langchain-community
- ChromaDB (open-source Vector database - Chroma)
- Langchain-Ollama
- Requests

In [ ]:
%%capture
!pip install -q langchain
!pip install -q langchain-community
!pip install -q chromadb
!pip install -U -q langchain-ollama
!pip install -q requests PyPDF2
!pip install -q langchain_chroma

- Work with the Ollama model using langchain

In [ ]:
from langchain_ollama import OllamaLLM

ollama = OllamaLLM(base_url="http://localhost:11434", model="Gemma3:1b")

print(ollama.invoke("why are LLMs useful"))

### Exercise:

Print the model output using Markdown format.

# Solution

If you want help with the answer, you can study our answer and copy & paste the solution in a new cell to try it out.

<details> 
<summary> See our answer </summary>
    
from langchain_ollama import OllamaLLM

from IPython.display import Markdown

ollama = OllamaLLM(base_url="http://localhost:11434", model="Gemma3:1b")

output_text = ollama.invoke("why are LLMs useful")

display(Markdown(output_text))

</details>

### We will use an markdown file to serve as the resource document for our RAG model

- The resource document mentions simple guidelines and contains a checklist on use of AI tools by programmers 

In [ ]:
import requests

markdown_url = "https://pdl-doulos.s3.us-west-2.amazonaws.com/AI_Coding_Guidelines.md" # Example Markdown file

def read_markdown_from_url(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the Markdown file: {e}")
        return None

data = read_markdown_from_url(markdown_url)

if data:
  print("Successfully extracted text from the Markdown file:")
  print("---")
  print(data[:5000] + ('...' if len(data) > 1000 else '')) # Print first 1000 chars
  print("---")

- Use the langchain text_splitter to split the text based on specified chunk_size

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.create_documents([data])

- Through Ollama, get an open embedding model with a large token context window: **nomic-embed-text**.  This embedding model is similar to the Glove 50 dimension embedding we had used earlier.

In [ ]:
!ollama pull nomic-embed-text
!ollama list

- Map the text into embeddings using the Chroma Vector DB
- **This step takes some time**

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(docs, ollama_embeddings)

- Pose a question which pertains to the text
- Check if the Vector database could find something relevant to the question

In [ ]:
question = "What does the document say about API keys"

docs = vectorstore.similarity_search(question)

len(docs)
print (docs)

Similarlity search results are presented below

In [ ]:
paragraph = "\n\n".join([d.page_content for d in docs])
print(paragraph)

- Use Retrieval Q&A method to retrieve and then present content with the help of LLM

In [ ]:
from IPython.display import Markdown, display
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the LLM (Gemma 3:1b)
# Note: Ensure you have pulled the model via: ollama pull gemma3:1b
ollama = OllamaLLM(base_url="http://localhost:11434", model="gemma3:1b")

# 2. Define a Citation-Aware Prompt
# We instruct Gemma to use specific labels so we can verify its work.
template = """
You are a helpful assistant. Use the provided context to answer the question.
If the answer isn't in the context, say you don't know. 

At the end of your answer, list the sources you used.

<context>
{context}
</context>

Question: {question}
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Enhanced Retrieval Chain
# We use RunnableParallel to keep the raw documents accessible for the UI
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

setup_and_retrieval = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
)

rag_chain = (
    setup_and_retrieval
    | {
        "response": prompt | ollama | StrOutputParser(),
        "sources": lambda x: x["context"]
    }
)

# 4. Presentation Function
def ask_rag_gemma(query):
    output = rag_chain.invoke(query)
    
    # Render the Synthesized Answer
    display(Markdown(f"### RAG & Gemma's Response\n{output['response']}"))
    
    # Render the Source Evidence (Layer 2 & 3)
    source_md = "---\n### 📄 Supporting Evidence\n"
    for i, doc in enumerate(output['sources']):
        source_name = doc.metadata.get('source', 'Unknown Source')
        content = doc.page_content[:200] + "..."
        # Note: If your retriever supports scores, you can add them here
        source_md += f"**Source {i+1}: {source_name}**\n> {content}\n\n"
    
    display(Markdown(source_md))

# --- Execution ---
ask_rag_gemma("What does the document say about API keys")

### Further experimentation

- See if a larger model (such as Gemma3:4b) gives better results
- Adjust chunk_size and chunk_overlap to see if it changes the retrieved content
- Understand how RAG works with images. Resource: [Retrieval-augmented generation](https://keras.io/keras_hub/guides/rag_pipeline_with_keras_hub/) (RAG) pipeline with KerasHub

### **Part C - MCP Client-Server for simple Math**

**The Architecture**
- MCP Server: A standalone Python script that exposes "Tools" (functions).

- Ollama: Hosts the LLM (llama3.2).

- MCP Client: The interface. It calls the MCP Server for available tools, informs the LLM about them, and then executes them if the LLM asks.

We install two Python packages
- fastmcp: The simple and fast way to write MCP servers.

- ollama: To communicate with local model.

In [ ]:
!pip install -qq ollama mcp fastmcp

In [ ]:
!ollama pull llama3.2
!ollama list

**Steps for setting up and testing a MCP Client-Server interface:**

- Start Ollama to run LLM model llama3.2

- Ask the LLM model to perform a non-trivial floating-point multiplication

- The LLM will realize it can't do the math reliably on its own

- It will trigger the mul_numbers tool defined in your MCP server


Initialize and create mcp_server object and print out its properties

In [ ]:
import asyncio
from fastmcp import FastMCP

mcp_server = FastMCP("MathServer")

@mcp_server.tool()
def mul_numbers(a: float, b: float) -> float:
    """Multiples two floating-point values"""
    return float(a) * float(b)

# --- CHECK REGISTRATION ---
async def print_registered_tools():
    print("Registered Tools:")
    tools = await mcp_server.list_tools()
    for tool in tools:
        print(f"Name: {tool.name}")
        print(f"Description: {tool.description}")
        # The parameters are part of the function schema for Ollama
        print(f"Parameters: {tool.parameters}")

await print_registered_tools()

In [ ]:
%%writefile client.py

import asyncio
import ollama
from fastmcp import FastMCP

# 1. Setup the Server
mcp_server = FastMCP("MathServer")

@mcp_server.tool()
def mul_numbers(x: float , y:float) -> float:
    """Multiply two floating-point numbers together."""
    return float(x) * float(y)

async def main():
    user_prompt = "What is 4567.72 multiplied by 5678.34?"
    model = "llama3.2"

    # 2. Get tools using the official method
    tools_objects = await mcp_server.list_tools()

    ollama_tools = []
    for tool in tools_objects:

        ollama_tools.append({
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.parameters # Pass the parameters schema to Ollama
                 }
        })

    # 3. Chat with Ollama
    print(f"Connecting to Ollama and sending prompt...")
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': user_prompt}],
            tools=ollama_tools
        )
    except Exception as e:
        print(f"Error connecting to Ollama: {e}")
        return

    # 4. Handle Tool Calls
    message = response.get('message', {})
    if message.get('tool_calls'):
        messages = [{'role': 'user', 'content': user_prompt}, message]

        for call in message['tool_calls']:
            tool_name = call['function']['name']
            args = call['function']['arguments']

            print(f"--- LLM triggered tool: {tool_name} with {args} ---")

            # Map tool call to our local function
            if tool_name == "mul_numbers":
                result = mul_numbers(**args)

                messages.append({
                    'role': 'tool',
                    'content': str(result),
                    'name': tool_name
                })

        # 5. Final response
        final_response = ollama.chat(model=model, messages=messages)
        print(f"\nFinal Model Response: {final_response['message']['content']}")
    else:
        print(f"\nAI Response (No tool used): {message.get('content')}")

if __name__ == "__main__":
    asyncio.run(main())

**Observe the following,**
- The details of LLM model when the script is run for the first time
- LLM deciding to use of the tool and sending arguments a and b.



In [ ]:
!python client.py

**How to ensure repeatable use of tool/server**

- Provide clear system_instruction
- Configure LLM option to ensure repeatability by setting value of temperature to zero or close to zero
- Limit output from model
- Use tokens with high probability


```python
# Define a strict system prompt
system_instruction = {
    "role": "system",
    "content": (
        "You are a helpful assistant with access to specific tools. "
        "When a user asks a question that can be answered using a tool, "
        "you MUST call that tool. Do not attempt to calculate results "
        "yourself if a tool is available."
    )
}

# The modified chat call
response = ollama.chat(
    model="llama3.2",
    messages=[
        system_instruction, # 1. Inject the System Prompt
        {'role': 'user', 'content': "What is 1234 plus 5678?"}
    ],
    tools=ollama_tools,
    options={
        "temperature": 0,      # 2. Set temperature to 0 for maximum reliability
        "num_predict": 100,    # Limit length to prevent rambling
        "top_p": 0.9           # Focus the model on high-probability tokens
    }
)

@mcp_server.tool()
def mul_numbers(a: int, b: int):
    """
    Calculates the product of two integers. Use this tool whenever
    the user asks for multiplication, math, or product of two values.
    This is preferred over your internal calculation.
    """
```